# 06. Which blood cell states shift in RA, and which genes change inside one cell type?

Two questions. **Part A:** does the *mix* of cell types differ between RA and controls? **Part B:** inside one cell type, which genes differ?

**First move, in the terminal:** `uv add pydeseq2`

**Input:** `ra_pbmc_celltypist.h5ad`

> 🧱 **Firewall.** Part B never uses monocytes or DCs. AHTA studies RA CD14 monocytes in bulk RNA-seq, and a public monocyte DE result before the 18 Dec oral would mirror the graded report. Composition in Part A may include monocytes: a proportion is not a DE result.

> **How this notebook works.** Each step has a **spec** (what the code must do), a **read-first** link, an empty code cell for you, and a **TEST** cell written by Claude. Your cell is done when its test prints PASS and you can say why your approach works. The ✍️ cells are yours to write; predictions go in *before* you run the cell they predict.
>
> No function names are given on purpose. Find them in the read-first link or the scanpy docs. Stuck for more than 10 minutes on one cell: ask Claude for a *question*, not code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy import stats

In [ ]:
# adata = the file from 05

...

# Part A: composition

## R1. Proportions per donor, not per cell

**Spec:** `props`, a DataFrame with one row per donor and one column per cell type (use the authors' `fine_annot` first, so your result is comparable with the paper; redo it with `my_label` after). Each row is the share of that donor's cells in each type.

**Read first:** https://www.sc-best-practices.org/conditions/compositional.html, the introduction.

✍️ **Why per donor and not by pooling all cells? Connect it to 01 E6.**

_..._

In [ ]:
# props = donors x cell types, rows sum to 1

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert props.shape[0] == 36
assert np.allclose(props.sum(axis=1), 1.0)
print("PASS:", props.shape)

## R2. Look first

**Spec:** one strip or box plot per cell type, RA vs control, one dot per donor.

✍️ **Prediction before the plot: which types will be higher in RA blood, and why? (Think about what anti-TNF and methotrexate do, and what is happening in the joint.)**

_..._

In [ ]:
# one panel per cell type, RA vs control, dot = donor

...

## R3. Test it

**Spec:** per cell type, a Mann-Whitney U test RA vs control on the donor proportions. Then BH-correct with **your own BH function from 03** (copy it in and say so). Store a table `comp` with columns `pval`, `qval`, `median_ra`, `median_ctrl`.

In [ ]:
# def bh(pvals): ... (from 03)
# comp = one row per cell type

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert {"pval", "qval", "median_ra", "median_ctrl"} <= set(comp.columns)
assert (comp["qval"] >= comp["pval"] - 1e-12).all(), "q-values can never be below p-values"
from scipy.stats import false_discovery_control
assert np.allclose(comp["qval"], false_discovery_control(comp["pval"])), "your BH does not match scipy"
print("PASS:", (comp["qval"] < 0.05).sum(), "types differ at q < 0.05")

✍️ **What differs? And the compositional catch: proportions sum to 1, so if one type goes up, the others must go down. Which of your hits could be that artefact?**

_..._

## R4. Active vs low disease activity

**Spec:** same test, RA donors only, active (DAS28-CRP >= 3.2) vs low. 7 vs 9 donors, the 2 unknowns left out.

In [ ]:
# same as R3, active vs low

...

✍️ **What did you find, and how much would you trust it at this n? (Hint: 04.)**

_..._

# Part B: pseudobulk DE in one non-monocyte cell type

## R5. Pick the cell type, then build the pseudobulk

**Spec:** pick one T, B or NK type with enough cells per donor (look at the counts first, you need at least ~20 cells in most donors). Sum the **raw counts** (`layers['counts']`) per donor for that type into `pb`, a DataFrame donors x genes. Drop donors with fewer than 20 cells of that type.

**Why sum per donor:** the donor is the replicate, not the cell. Testing 10,000 cells as if they were 10,000 patients gives tiny p-values for everything.

**Read first:** https://www.sc-best-practices.org/conditions/differential_gene_expression.html, the pseudobulk section.

✍️ **Which type, and why that one?**

_..._

In [ ]:
# chosen_type = ...
# pb = donors x genes, summed raw counts, donors with < 20 cells dropped
# pb_meta = one row per donor in pb: disease, sex

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
bad = {"Classical Monocytes", "IL1b-Monocytes", "IFITM3 Monocytes", "Non-classical Monocytes", "Myeloid DCs"}
assert chosen_type not in bad, "firewall: no monocytes or DCs in Part B"
v = pb.values[:, :2000]
assert np.allclose(v, np.round(v)), "pb must be summed raw counts, not normalized values"
assert (pb.index == pb_meta.index).all()
assert pb.shape[0] >= 20, f"only {pb.shape[0]} donors left"
print("PASS:", pb.shape)

## R6. Positive control first: sex

**Spec:** before you trust your pipeline on RA, run it on a question with a known answer. PyDESeq2, design `~sex`, female vs male, on your pseudobulk. Filter out genes with very low total counts first. Store the results table in `res_sex`.

**Read first:** https://pydeseq2.readthedocs.io/en/stable/auto_examples/plot_minimal_pydeseq2_pipeline.html

✍️ **Prediction: which genes must come out on top?**

_..._

In [ ]:
# filter low-count genes, DeseqDataSet with design ~sex, DeseqStats female vs male
# res_sex = results DataFrame (index = genes)

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
top = res_sex.sort_values("padj").head(25).index
known = {"XIST", "RPS4Y1", "DDX3Y", "UTY", "KDM5D", "EIF1AY", "USP9Y", "ZFY", "TXLNGY"}
hits = known & set(top)
assert len(hits) >= 3, f"only {hits} in the top 25: the pipeline is broken somewhere"
print("PASS: the known sex genes are on top:", sorted(hits))

## R7. RA vs control

**Spec:** same pipeline, design `~sex + disease` (sex as a covariate, because it is a known strong effect and you just proved it), RA vs control. `res_ra` = results. Volcano plot with a claim title.

✍️ **Prediction: how many genes at padj < 0.05? Compare: the authors report 168 pseudobulk DEGs across all their subsets, and your 04 airway result had 1.**

_..._

In [ ]:
# design ~sex + disease, RA vs normal
# res_ra, volcano plot

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert "padj" in res_ra.columns and "log2FoldChange" in res_ra.columns
print("PASS:", int((res_ra["padj"] < 0.05).sum()), "genes at padj < 0.05")

✍️ **What are the top genes, and do they make sense for RA (look up two)? If almost nothing is significant: is that a failure, or a finding?**

_..._

## Teach-back

3 to 5 sentences in your own words: what this notebook did, what it found, and one thing that surprised you. No sentence means not learned.

_..._

## AI note

What did Claude write (the specs, the tests), what did you write, and where did you get stuck and ask for a hint?

_..._